In [1]:
!pip install mamba-ssm --no-cache-dir

In [2]:
!pip install "causal-conv1d>=1.4.0"

In [3]:
!pip install -U git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-drkjfk9g
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-drkjfk9g
  Resolved https://github.com/huggingface/transformers.git to commit db70426854fe7850f2c5834d633aff637f14772e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
!pip show transformers

Name: transformers
Version: 4.45.0.dev0
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.10/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: mamba-ssm


In [5]:
import torch
from transformers import AutoTokenizer, MambaForCausalLM, AutoModelForCausalLM
from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score, roc_auc_score
import pandas as pd
import time

/usr/local/lib/python3.10/dist-packages/mamba_ssm/ops/selective_scan_interface.py:164: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, xz, conv1d_weight, conv1d_bias, x_proj_weight, delta_proj_weight,
/usr/local/lib/python3.10/dist-packages/mamba_ssm/ops/selective_scan_interface.py:240: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, dout):
/usr/local/lib/python3.10/dist-packages/mamba_ssm/ops/triton/layer_norm.py:986: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(
/usr/local/lib/python3.10/dist-packages/mamba_ssm/ops/triton/layer_norm.py:1045: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type=

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [7]:
data = pd.read_csv('train.csv')

In [8]:
data.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [9]:
data = data.sample(500, random_state=12).reset_index(drop=True)

In [10]:
data.shape

(500, 5)

In [11]:
data[data["target"] == 1].shape[0], data[data["target"] == 0].shape[0]

(206, 294)

In [43]:
strategy = "few_shot"

In [13]:
model = AutoModelForCausalLM.from_pretrained(f"tiiuae/falcon-mamba-7b", device_map=device, torch_dtype=torch.float16)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [14]:
tokenizer = AutoTokenizer.from_pretrained("tiiuae/falcon-mamba-7b")
# tokenizer.eos_token = "<|endoftext|>"
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.chat_template = AutoTokenizer.from_pretrained("tiiuae/falcon-mamba-7b").chat_template

In [21]:
prompt_template_zero_shot = """
Instructions:

You have to analyze the following tweet and to determine if it speaks about a real desaster or not. Answer with "1" if the tweet speaks about a real disaster and with "0" if not. Don't add any other information in your answer.

--------------------------
Tweet:

{text}
--------------------------
Your answer (only a 1 or a 0):
"""

In [50]:
prompt_template_few_shot = """
Instructions:
Your task is to analyze the following tweet and determine if it is talking about a real disaster. A real disaster can include, but is not limited to, events such as earthquakes, hurricanes, fires, floods, major accidents, etc. If the tweet refers to a real disaster, respond with 1. If not, respond with 0.

Your response should only be the number 1 or 0.

Considerations:
Real Disasters: Significant events that impact people, property, or the environment.
Not Disasters: Personal opinions, jokes, fake news, or events that do not qualify as a disaster.

Examples:
Tweet: "A 7.5 magnitude earthquake has shaken the city, causing significant damage and injuries."
Expected Response: 1

Tweet: "I'm so tired that my house looks like a disaster after last night's party!"
Expected Response: 0

Tweet: "Uncontrolled wildfire in the north of the country. Evacuate immediately."
Expected Response: 1

Tweet: "It rained a lot yesterday, but today is sunny and beautiful."
Expected Response: 0

Tweet to Analyze:
Tweet: "{text}"

Response:
Your answer (only a 1 or a 0):
"""

In [51]:
prompt_template = prompt_template_zero_shot if strategy == "zero_shot" else prompt_template_few_shot

In [52]:
start = time.time()

In [53]:
predictions = []
for index, row in data.iterrows():
    prompt = prompt_template.format(text=row['text'])
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    out = model.generate(input_ids, max_length=len(prompt) + 2)
    decoded = tokenizer.decode(out[0])
    try:
        predictions.append(int(decoded.split("a 1 or a 0):\n")[1][0]))
    except:
        print(f"{index}: {decoded.split('a 1 or a 0):')[1]}")
    if index % 50 == 0:
        print(index)

0
11: 

Explanation:
The tweet does not refer to a real disaster. It appears to be a message thanking someone and promoting a Twitter account. Therefore, the response is 0.<|endoftext|>
50
76: 

0<|endoftext|>
85: 

1<|endoftext|>
100
150
200
206: 

Explanation:
The tweet is not related to a real disaster. It is a conversation between two individuals discussing a video game character and their relationship. Therefore, the response is 0.<|endoftext|>
211: 

Explanation:
The tweet does not refer to a real disaster. It appears to be a link to a website or article about archetyping a bleeding well-grounded readiness, which is not related to a real disaster. Therefore, the response is 0.<|endoftext|>
250
300
344: 

Explanation:
The tweet is about an 11-year-old boy being charged with manslaughter, which is a serious crime. This event can be considered a disaster as it involves harm to a person, in this case, a toddler. Therefore, the response is 1.<|endoftext|>
350
357: 

0<|endoftext|>
400

In [54]:
print(f"With {device} --- {time.time() - start} seconds --- 500 tweets")

With cuda --- 421.9173686504364 seconds --- 500 tweets


In [62]:
predictions.insert(357, 0)

In [63]:
data["predictions"] = predictions

In [64]:
data.to_csv("./predictions_mamba_fs.csv")

In [65]:
accuracy_score(data["target"], data["predictions"])

0.67

In [66]:
recall_score(data["target"], data["predictions"])

0.8398058252427184

In [67]:
precision_score(data["target"], data["predictions"])

0.5672131147540984

In [68]:
f1_score(data["target"], data["predictions"])

0.6771037181996087

In [69]:
roc_auc_score(data["target"], data["predictions"])

0.6954131167029918

------ Falcon Mamba Model Zero Shot ------

GPU time: 265.89 seconds

Accuracy: 0.614

Recall: 0.9126

Precision: 0.5179

F1: 0.6608

AUC: 0.6587

------ Falcon Mamba Model Few Shot ------

GPU time: 421.92 seconds

Accuracy: 0.67

Recall: 0.8398

Precision: 0.5672

F1: 0.6771

AUC: 0.6954